# Day 33：高级 RAG Part 2 — BM25 + 向量混合检索、RRF、重排与父文档检索

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段 Day 33！今天我们将实战 `BM25Retriever`、`VectorRetriever`、`reciprocal_rank_fusion`（RRF）、`CrossEncoderReranker` 以及 `ParentDocumentRetriever`（Small-to-Big），构建工业级高准确率 RAG 检索流水线（`v0.3.0-day33`）。

## 1. 环境初始化与导入 `project_iterations/day_33` 模块
定位仓库根目录并将 `project_iterations/day_33` 加入 `sys.path`，导入本节核心检索组件。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day33_dir = repo_root / "project_iterations" / "day_33"
if str(day33_dir) not in sys.path:
    sys.path.insert(0, str(day33_dir))

from zhishu_agent import (
    BM25Retriever,
    CrossEncoderReranker,
    Document,
    ParentDocumentRetriever,
    VectorRetriever,
    reciprocal_rank_fusion,
)

print("已加载模块路径:", day33_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_33


## 2. BM25 稀疏检索 + Vector 稠密检索 + RRF 倒数排名融合
构建包含精确错误码与企业制度的文档集，对比 BM25 与 Vector 的召回差异，并通过 `reciprocal_rank_fusion(..., k=60)` 融合排名。

In [2]:
docs = [
    Document(
        id="runbook-502",
        page_content="【生产排障】错误码 ERR_NGX_502 处理手册：检查上游 Pod 是否 OOMKilled 及 keepalive_timeout。",
        metadata={"dept": "ops"},
    ),
    Document(
        id="arch-rag",
        page_content="【架构规范】智枢 RAG 采用 BM25 与向量双路召回，经 RRF 融合与 CrossEncoder 精排输出 Top-K。",
        metadata={"dept": "ai"},
    ),
    Document(
        id="hr-leave",
        page_content="【人事制度】入职满1年享有5天带薪年假，需提前3个工作日在 OA 系统申请。",
        metadata={"dept": "hr"},
    ),
    Document(
        id="fin-travel",
        page_content="【财务规范】一线城市差旅住宿标准为每晚600元以内，需提供电子发票。",
        metadata={"dept": "finance"},
    ),
]

bm25 = BM25Retriever(docs)
vector = VectorRetriever(docs)

query = "ERR_NGX_502 网关报错排查"
bm25_hits = bm25.retrieve_with_scores(query, top_k=3)
vec_hits = vector.retrieve_with_scores(query, top_k=3)
rrf_hits = reciprocal_rank_fusion([bm25_hits, vec_hits], k=60)

print("BM25 召回:", [(d.id, s) for d, s in bm25_hits])
print("Vector 召回:", [(d.id, s) for d, s in vec_hits])
print("RRF 融合结果:", [(d.id, s) for d, s in rrf_hits])

BM25 召回: [('runbook-502', 4.787376), ('arch-rag', 0.688523)]
Vector 召回: [('runbook-502', 0.208666), ('hr-leave', 0.011919), ('fin-travel', -0.035176)]
RRF 融合结果: [('runbook-502', 0.032787), ('arch-rag', 0.016129), ('hr-leave', 0.016129), ('fin-travel', 0.015873)]


## 3. CrossEncoderReranker 精排
对 RRF 融合后的候选文档集合执行词法覆盖度 + 语义相似度 + 短语匹配联合精排。

In [3]:
reranker = CrossEncoderReranker()
candidate_docs = [doc for doc, _ in rrf_hits]
reranked = reranker.rerank(query, candidate_docs, top_k=2)

for rank, (doc, score) in enumerate(reranked, start=1):
    print(f"Top-{rank} | id={doc.id} | rerank_score={score:.4f} | {doc.page_content}")

Top-1 | id=runbook-502 | rerank_score=0.2772 | 【生产排障】错误码 ERR_NGX_502 处理手册：检查上游 Pod 是否 OOMKilled 及 keepalive_timeout。
Top-2 | id=arch-rag | rerank_score=0.0458 | 【架构规范】智枢 RAG 采用 BM25 与向量双路召回，经 RRF 融合与 CrossEncoder 精排输出 Top-K。


## 4. ParentDocumentRetriever（Small-to-Big 父文档检索）
将长文档切分为细粒度子切片建立混合索引，检索命中子切片后自动映射并去重返回完整父文档。

In [4]:
parent_docs = [
    Document(
        id="handbook-k8s",
        page_content=(
            "第一章 集群概述：智枢生产集群部署于多可用区。"
            "第二章 内存故障排查：当容器内存超过 limits 时会触发 OOMKilled（退出码 137），"
            "此时应使用 kubectl describe pod 查看 Last State 并调大 memory limit。"
            "第三章 网络策略：默认启用 Calico NetworkPolicy 隔离命名空间。"
        ),
        metadata={"source": "k8s_handbook.md"},
    ),
    Document(
        id="handbook-git",
        page_content="第一章 分支模型：功能开发统一从 main 拉取 feature 分支，提交遵循 Conventional Commits。",
        metadata={"source": "git_handbook.md"},
    ),
]

parent_retriever = ParentDocumentRetriever(chunk_size=70, chunk_overlap=15)
total_chunks = parent_retriever.add_documents(parent_docs)
print(f"已切分并索引子切片数量: {total_chunks}")

matched = parent_retriever.retrieve_with_matched_children("OOMKilled 退出码 137", top_k=1)
for parent_doc, child_doc, rrf_score in matched:
    print(f"命中子切片 ({child_doc.id}, rrf={rrf_score:.4f}): {child_doc.page_content}")
    print(f"返回父文档 ({parent_doc.id}, 长度={len(parent_doc.page_content)}): {parent_doc.page_content}")

已切分并索引子切片数量: 4
命中子切片 (handbook-k8s::chunk-1, rrf=0.0325): OMKilled（退出码 137），此时应使用 kubectl describe pod 查看 Last State 并调大 memory 
返回父文档 (handbook-k8s, 长度=173): 第一章 集群概述：智枢生产集群部署于多可用区。第二章 内存故障排查：当容器内存超过 limits 时会触发 OOMKilled（退出码 137），此时应使用 kubectl describe pod 查看 Last State 并调大 memory limit。第三章 网络策略：默认启用 Calico NetworkPolicy 隔离命名空间。
